# Python for AI — Class 6
### Functions: write it once, use it forever

You've spent five days learning Python's building blocks — variables, conditionals, loops, lists, tuples, dictionaries, sets. Today those blocks get packaged.

This is a bigger session than usual. Instead of spreading "functions" across three separate days, you're getting the whole picture in one sitting: defining functions, every way to pass arguments, scope, lambda functions, recursion, decorators, and generators. It's more to absorb, but it's one connected story instead of three disconnected fragments — and everything after today builds on it.

**How to use this notebook:** run each cell with the play button, or `Shift + Enter`.

Cells marked **BREAKS ON PURPOSE** are *supposed* to show a red error. Cells marked **WRONG ON PURPOSE** run fine and give the *wrong answer* — those are the dangerous ones. Don't fix either before class; that is the lesson.

---
# 1. The problem functions solve

Three rectangles, three area calculations.

In [ ]:
print("Area of rectangle 1:", 4 * 5)
print("Area of rectangle 2:", 3 * 7)
print("Area of rectangle 3:", 10 * 2)

The calculation — `width * height` — is copy-pasted three times. Copy-pasted code is code you now have to fix in three places if the formula ever changes.

**A function lets you write the logic once, give it a name, and reuse it.**

---
# 2. Defining and calling a function

Notice the shape: a line ending in a **colon**, an **indented block** underneath. Same shape as `if`, `for`, and `while` — you already know how to read this.

In [60]:
def area(width, height):
    return 5
    a = width * height
    return a
    # print(width * height)

# print(area(4, 5))
# print(area(3, 7))
# print(area(10, 2))

rect_1 = area(4, 5)

print("Area of rectangle 1:", rect_1)



Area of rectangle 1: 5


Break that first line down:

- `def` — "I am about to define a function"
- `area` — the name **you chose**, same rule as loop variable names on Day 3
- `(width, height)` — the **parameters**: named placeholders for values the function needs
- `:` then an indented block — the **body**, the code that runs every time you call it

`width` and `height` are **parameters** — the placeholders written in the `def` line. `4` and `5` in `area(4, 5)` are **arguments** — the actual values you hand over when you call it. People use the words loosely in conversation, but the distinction is worth having: a parameter is a name, an argument is a value.

Defining a function does not run it. Nothing gets printed until you **call** it — `area(4, 5)` — the same way `range(5)` on Day 3 did nothing on its own until a `for` loop consumed it.

---
# 3. `return` — sending a value back

`return` hands a value back to wherever the function was called, so you can store it, print it, or use it in another expression.

In [61]:
def add(a, b):
    return a + b

result = add(3, 4)
print(result)
print(add(3, 4) * 10)

7
70


### The classic mix-up: `print` inside a function isn't the same as `return`

`print` just displays something on the screen — it doesn't hand anything back to the code that called the function.

In [62]:
# WRONG ON PURPOSE - this looks like it works, but it hands back nothing
def add_and_print(a, b):
    print(a + b)      # displays the answer...

result = add_and_print(3, 4)
print("result is:", result)   # ...but result is None

7
result is: None


`add_and_print` printed `7`, which looks like success — but `result` is `None`. A function with no `return` statement (or a `return` with nothing after it) **implicitly returns `None`**. If you need the value again later, you must `return` it, not just `print` it.

`return` also **exits the function immediately** — the moment Python hits a `return`, it leaves, skipping anything written after it. Same idea as `break` leaving a loop on Day 3.

In [64]:
def check_sign(n):
    if n > 0:
        return "positive"
    if n < 0:
        return "negative"
    return False

print(check_sign(5))
print(check_sign(-3))
print(check_sign(0))

positive
negative
False


---
# 4. Parameters are matched by position

When you call a function with plain values, they fill the parameters **left to right, in order**.

In [2]:
def greet(name, greeting):
    print(greeting, name)

greet("Ali", "Hello")

Hello Ali


In [3]:
# WRONG ON PURPOSE - the arguments are swapped, so the roles are swapped too
greet("Hello", "Ali")

Ali Hello


No error — `greeting` is now `"Ali"` and `name` is now `"Hello"`. Positional arguments only work if you get the order right; there's nothing checking that "Ali" was meant to be a name.

---
# 5. Default arguments

Give a parameter a value in the `def` line, and callers can skip it.

In [ ]:
def greet(name, greeting="Hello"):
    print(greeting, name)

greet("Ali")               # uses the default
greet("Ali", "Salaam")     # overrides it

**A default parameter can't come before a required one** — Python needs to know, just by reading the `def` line, which arguments are optional.

In [ ]:
# BREAKS ON PURPOSE - a default parameter can't come before a required one
def greet(greeting="Hello", name):
    print(greeting, name)

### The mutable default trap

This is one of the most well-known gotchas in Python, and it connects straight back to Day 4's copy trap: a default value is created **once**, when the function is defined — not fresh on every call.

In [4]:
# WRONG ON PURPOSE - the same list is reused across every call
def add_item(item, cart=[]):
    cart.append(item)
    return cart

print(add_item("pen"))     # ['pen'] - looks right
print(add_item("book"))    # ['pen', 'book'] - wait, where did 'pen' come from?

['pen']
['pen', 'book']


`cart=[]` only runs **once**, at `def` time. Every call that doesn't supply its own `cart` shares that *same* list — exactly like `b = a` on Day 4 gave you a second sticker on the same list, not a fresh one.

The fix: default to `None`, and create the real empty list *inside* the function body, where it's rebuilt on every call.

In [5]:
def add_item(item, cart=None):
    if cart is None:
        cart = []
    cart.append(item)
    return cart

print(add_item("pen"))
print(add_item("book"))

['pen']
['book']


> **Rule of thumb:** never use a mutable value (`[]`, `{}`, `set()`) as a default argument. Use `None` and build the real thing inside the function.

---
# 6. Keyword arguments

Instead of relying on position, you can name each argument when you call the function. Order stops mattering, and the call documents itself.

In [ ]:
def describe_pet(name, animal_type, age):
    print(f"{name} is a {age}-year-old {animal_type}")

describe_pet("Rex", "dog", 3)                        # positional
describe_pet(name="Rex", animal_type="dog", age=3)   # keyword
describe_pet(age=3, name="Rex", animal_type="dog")   # order doesn't matter now

You can mix the two, but **every positional argument must come before every keyword argument** — once you start naming values, you can't go back to unnamed ones.

In [ ]:
describe_pet("Rex", age=3, animal_type="dog")     # fine - positional first, then keyword

In [ ]:
# BREAKS ON PURPOSE - a positional argument after a keyword argument
describe_pet(name="Rex", "dog", 3)

---
# 7. `*args` — any number of positional arguments

Sometimes you don't know in advance how many values a function will get. `*args` collects every extra positional argument into a **tuple**.

In [6]:
def total(*numbers):
    print(numbers, type(numbers))
    return sum(numbers)

print(total(1, 2, 3))
print(total(1, 2, 3, 4, 5))
print(total())

(1, 2, 3) <class 'tuple'>
6
(1, 2, 3, 4, 5) <class 'tuple'>
15
() <class 'tuple'>
0


`args` isn't a keyword — it's just a name, and the convention everyone uses. The `*` is what actually matters: it tells Python "gather up whatever positional arguments are left over." Same idea as Day 3's "the name is yours to pick, `for letter in name` just reads better than `for x in name`."

### Real use case: a POS till

A shop's point-of-sale (POS) till has no idea how many items the next customer will buy — one, or thirty. `*args` handles any number.

(You've actually been using `*args` since Day 1: `print()` and `max()` both accept any number of values.)

In [ ]:
#till

def ring_up(*prices):
    subtotal = sum(prices)
    tax = subtotal * 0.18                 # 18% sales tax
    return round(subtotal + tax, 2)

print(ring_up(250))                       # a customer buying one item
print(ring_up(250, 1200, 80, 45))         # a customer with a full basket

---
# 8. `**kwargs` — any number of keyword arguments

`**kwargs` is the keyword-argument version: it collects every extra `name=value` pair into a **dictionary**.

In [7]:
def print_profile(**details):
    print(details, type(details))
    for key, value in details.items():
        print(key, "->", value)

print_profile(name="Ali", age=25, city="Karachi")

{'name': 'Ali', 'age': 25, 'city': 'Karachi'} <class 'dict'>
name -> Ali
age -> 25
city -> Karachi


That `.items()` loop is exactly Day 5's pattern — `**kwargs` is just a dictionary that arrived through a function call instead of a `{}` literal.

### Combining regular parameters, `*args`, and `**kwargs`

When you use all three, they must appear in this order in the `def` line: **regular parameters, then `*args`, then `**kwargs`.**

In [8]:
def order_summary(customer, *items, **extras):
    print("Customer:", customer)
    print("Items:", items)
    print("Extras:", extras)

order_summary("Ali", "pizza", "coke", discount=10, gift_wrap=True)

Customer: Ali
Items: ('pizza', 'coke')
Extras: {'discount': 10, 'gift_wrap': True}


In [ ]:
# BREAKS ON PURPOSE - **kwargs must be the last parameter
def broken_order(customer, **extras, *items):
    pass

### Real use case: placing an online order

An e-commerce checkout has sensible defaults — standard shipping, no gift wrap — and each customer changes only the few options they care about. `**kwargs` lets one function accept any of those options without listing every possible one in the `def` line.

Many libraries you'll use later, including the AI libraries in Month 2, take optional keyword settings exactly like this.

In [ ]:
def create_order(customer, *items, **options):
    order = {
        "customer": customer,
        "items": list(items),
        "shipping": "standard",               # defaults
        "gift_wrap": False,
    }
    for key, value in options.items():        # override or add whatever the customer chose
        order[key] = value
    return order

print(create_order("Ali", "Headphones"))
print(create_order("Fatima", "Laptop", "Mouse", shipping="express", gift_wrap=True, coupon="EID20"))

---
# 9. Scope: local vs. global

A variable created **inside** a function is **local** — it exists only while that function is running, and disappears the moment it returns.

In [9]:
def set_score():
    score = 100
    print("Inside the function:", score)

set_score()

Inside the function: 100


In [10]:
# BREAKS ON PURPOSE - score only ever existed inside set_score
print("Outside the function:", score)

NameError: name 'score' is not defined

**NameError** — `score` was never created out here. This is a feature, not a bug: it means two different functions can both use a variable called `score` and never interfere with each other.

A variable created **outside** every function is **global**, and functions can freely *read* one.

In [11]:
total_sales = 0

def show_sales():
    print("Sales so far:", total_sales)     # reading a global works fine

show_sales()

Sales so far: 0


**But assigning to a global-named variable inside a function creates a new local one instead** — Python decides a name is local for the *entire* function the moment it sees an assignment to it anywhere inside that function, even below where you tried to read it.

In [12]:
# BREAKS ON PURPOSE - Python treats total_sales as local because of the assignment below,
# so reading it on the right-hand side fails before that assignment ever runs
total_sales = 0

def add_sale(amount):
    total_sales = total_sales + amount
    print(total_sales)

add_sale(50)

UnboundLocalError: cannot access local variable 'total_sales' where it is not associated with a value

**UnboundLocalError.** To actually modify a global variable from inside a function, say so explicitly with `global`:

In [13]:
total_sales = 0

def add_sale(amount):
    global total_sales
    total_sales = total_sales + amount
    print(total_sales)

add_sale(50)
add_sale(30)
print("Final total:", total_sales)

50
80
Final total: 80


### Why you should reach for `global` rarely

`global` works, but treat it as a last resort. The idea: **don't let a function secretly reach out and change things outside itself. Hand it what it needs, and let it hand back the result.**

Think of a function as a chef in a kitchen:

- **The clean way (parameters + `return`):** you hand the chef the ingredients, and the chef hands you back a finished dish. You can see exactly what went in and what came out.
- **The `global` way:** the chef walks out of the kitchen, goes into your fridge, and changes what's in it without telling you. Later you open the fridge, something is missing, and you have no idea which chef did it or when.

Here's the same job done both ways.

In [ ]:
# The global way - the function secretly changes something outside itself
total = 0

def add_sale(amount):
    global total
    total = total + amount

add_sale(50)          # nothing on this line tells you `total` just changed
print(total)

In [ ]:
# The clean way - what goes in and what comes out are both visible
def add_sale(total, amount):
    return total + amount

total = 0
total = add_sale(total, 50)    # takes total and 50, the result goes back into total
print(total)

Both print `50`. The difference is what you can see.

In the first version, reading `add_sale(50)` gives no hint that `total` changed — you'd have to open the function and read its insides to find out. In the second, the call itself shows everything: it takes `total` and `50`, and the result goes back into `total`.

**Why globals cause more bugs than they fix:** once several functions can all change the same global variable, a wrong value could have come from any of them, and you end up hunting through every function to find out who changed it. With parameters and `return`, each function only touches what you hand it — so when something goes wrong, there are far fewer places to look.

---
# 10. Lambda — a function in one line

A `lambda` is a function with no name, restricted to a single expression, whose result is returned automatically.

In [ ]:
add = lambda a, b: a + b
print(add(3, 4))

Compare it to the `def` version — same job, different packaging:

```python
def add(a, b):
    return a + b
```

Read `lambda a, b: a + b` as **"a function of `a` and `b`, that returns `a + b`."** No `def`, no name, no `return` keyword — just parameters, a colon, and one expression whose value comes back automatically. There's no block of statements inside a lambda; if you need more than one expression, you need `def`.

### The real reason you'll use lambda: as a throwaway argument to another function

Writing `add = lambda a, b: a + b` and giving it a name, like above, is not what lambda is actually for — if it needs a name, just use `def`. Lambda earns its place when a function needs a small piece of logic **just once**, as an argument, and naming it separately would be more ceremony than the logic deserves.

In [20]:
students = [("Ali", 85), ("Fatima", 92), ("Hassan", 78)]

students.sort(key=lambda student: student[1])   # sort by the second item (marks)
print(students)

[('Hassan', 78), ('Ali', 85), ('Fatima', 92)]


`key=` tells `.sort()` **what to compare**, not what order to put things in. For every item, Python calls the lambda and sorts by whatever it returns — here, `student[1]`, the marks. Without `key=`, `.sort()` would try to compare whole tuples and you'd have no control over which field mattered.

In [26]:
students = [("Ali", 85), ("Fatima", 92), ("Hassan", 78)]

students.sort(key=lambda student: student[0])              # sort by name
print(students)

students.sort(key=lambda student: student[1], reverse=True)  # highest marks first
print(students)

print(max(students, key=lambda student: student[1]))        # the same idea with max()

print("Lambda:", (lambda student: student[0]))
# print("Lambda:", (lambda student: student[0])(students) )


[('Ali', 85), ('Fatima', 92), ('Hassan', 78)]
[('Fatima', 92), ('Ali', 85), ('Hassan', 78)]
('Fatima', 92)
Lambda: <function <lambda> at 0x10e66cf40>


That last line should look familiar — it's the same `key=` pattern Day 5 used with `max(counts, key=counts.get)` to find the most common word. `counts.get` and `lambda word: counts[word]` do the same job; `.get` just happened to already exist.

> **When *not* to use lambda:** if the logic needs more than one expression, needs a name to be readable, or gets reused in more than one place — write a regular `def` function instead. A lambda that needs a comment to explain it should have been a `def`.

### Real use cases: an online store's catalogue and sales report

Lambda's everyday job is telling `sorted()`, `.sort()`, `max()` and `min()` which part of each item to look at. Every online store does this constantly: "sort by price", "best sellers", "running low on stock".

In [ ]:
products = [
    {"name": "Laptop", "price": 150000, "stock": 5},
    {"name": "Mouse", "price": 1500, "stock": 42},
    {"name": "Monitor", "price": 45000, "stock": 2},
]

print("Cheapest first:")
for p in sorted(products, key=lambda p: p["price"]):
    print(" ", p["name"], p["price"])

print("Most expensive:", max(products, key=lambda p: p["price"])["name"])
print("Reorder soon:  ", min(products, key=lambda p: p["stock"])["name"])

In [ ]:
# Which products actually brought in the most money?
sales = [
    {"product": "Laptop", "price": 150000, "units_sold": 4},
    {"product": "Mouse", "price": 1500, "units_sold": 120},
    {"product": "Monitor", "price": 45000, "units_sold": 9},
    {"product": "Keyboard", "price": 3500, "units_sold": 60},
]

by_revenue = sorted(sales, key=lambda s: s["price"] * s["units_sold"], reverse=True)

for s in by_revenue:
    print(s["product"], s["price"] * s["units_sold"])

Two things worth noticing:

- The lambda doesn't have to just pick out a field — it can **calculate** something. Here it sorts by revenue, `price × units_sold`, which isn't stored anywhere.
- The Mouse sold the most units (120) but brought in the *least* money. "Best seller" depends on what you sort by — and choosing the right `key` is a business decision, not just a coding one.

---
# 11. Recursion — a function that calls itself

A recursive function solves a problem by solving a **smaller version of the same problem**, and calling itself to do it.

Every recursive function needs two parts:

1. A **base case** — the smallest version of the problem, simple enough to answer directly, with no further recursive call.
2. A **recursive case** — the function calling itself with a smaller piece of the problem, moving toward the base case.

Miss the base case, and it's Day 3's infinite loop again — except this time it's an infinite chain of function calls.

### Start simple: a countdown

In [ ]:
def countdown(n):
    if n == 0:                 # base case - stop here
        print("Done!")
        return
    print(n)
    countdown(n - 1)           # recursive case - the same job, one smaller

countdown(3)

Follow it call by call:

```
countdown(3)  prints 3, then calls countdown(2)
countdown(2)  prints 2, then calls countdown(1)
countdown(1)  prints 1, then calls countdown(0)
countdown(0)  base case: prints "Done!" and stops
```

Each call does one small piece of the work — print one number — then hands the rest of the job to a smaller copy of itself.

### Adding up 1 to n

In [ ]:
def sum_to(n):
    if n == 0:                  # base case - the sum of nothing is 0
        return 0
    return n + sum_to(n - 1)    # n, plus the sum of everything below it

print(sum_to(3))
print(sum_to(10))

```
sum_to(3) = 3 + sum_to(2)
          = 3 + 2 + sum_to(1)
          = 3 + 2 + 1 + sum_to(0)
          = 3 + 2 + 1 + 0
          = 6
```

### Reversing a word

In [43]:
def reverse(text):
    if text == "":                          # base case - nothing left to reverse
        return ""
    return reverse(text[1:]) + text[0]      # reverse the rest, then put the first letter last



print(reverse("hello"))

olleh


### The classic: factorial

`5!` ("5 factorial") means `5 × 4 × 3 × 2 × 1`. It has exactly the same shape as `sum_to` — multiply instead of add, and the base case returns `1` instead of `0`.

In [27]:
def factorial(n):
    if n == 0:                     # base case - stops the recursion
        return 1
    return n * factorial(n - 1)    # recursive case - a smaller problem

print(factorial(5))

120


Trace `factorial(3)` by hand — it has to go all the way down to the base case before anything can be multiplied:

```
factorial(3)
= 3 * factorial(2)
    = 3 * (2 * factorial(1))
        = 3 * (2 * (1 * factorial(0)))
            = 3 * (2 * (1 * 1))         <- base case reached, returns 1
        = 3 * (2 * 1)                   <- factorial(1) returns 1
    = 3 * 2                             <- factorial(2) returns 2
= 6                                     <- factorial(3) returns 6
```

### The call stack

Each call to `factorial` pauses and waits for the call it made to finish, before it can compute its own answer. Python keeps track of every paused, waiting call in a structure called the **call stack** — literally a stack, like Day 4's `append`/`pop`: the most recent call is the first one to finish and get popped off.

Forget the base case, and that stack never stops growing:

In [28]:
# BREAKS ON PURPOSE - no base case, so it never stops calling itself
def broken_factorial(n):
    return n * broken_factorial(n - 1)

broken_factorial(5)

RecursionError: maximum recursion depth exceeded

**RecursionError: maximum recursion depth exceeded.** This is recursion's version of Day 3's infinite loop — except an infinite `while` loop just hangs forever, while Python tracks the call stack's size and refuses to let it grow without limit, so a broken recursion crashes cleanly instead of freezing your program.

### Recursion isn't just for numbers

The "smaller version of the same problem" can be a smaller list, a smaller string, a smaller anything.

In [ ]:
def sum_list(numbers):
    if not numbers:                      # base case - an empty list sums to 0
        return 0
    return numbers[0] + sum_list(numbers[1:])   # first item + the sum of the rest

print(sum_list([1, 2, 3, 4, 5]))
print(sum_list([]))

Each call peels off `numbers[0]` and hands the rest, `numbers[1:]`, to itself. The list gets one item shorter every call, so it's guaranteed to eventually hit the base case: an empty list.

In [33]:
# Explain // operation
# In Python, the `//` operator is used for floor division. 
# It divides two numbers and returns the largest integer less than or equal to the result. 
# This means that it effectively "rounds down" to the nearest whole number.
# Example:
a = 1234 // 10
print(a)  # Output: 123, because 1234 divided by 10 is 123.4, and the floor division returns 123.


123


In [34]:
def sum_digits(n):
    if n < 10:              # base case - a single digit sums to itself
        return n
    return n % 10 + sum_digits(n // 10)    # last digit + the sum of the rest

print(sum_digits(1234))    # 1 + 2 + 3 + 4

10


### Recursion vs. a loop — the same answer, two shapes

Anything recursion can do, a loop can also do — usually with less overhead, since every recursive call takes up its own space on the call stack, while a loop just keeps reusing the same space.

In [35]:
def factorial_loop(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

print(factorial_loop(5))

120


Recursion earns its place when a problem is *naturally* described in terms of a smaller version of itself — nested data like a business's expense categories is the clearest example, and you'll see one at the end of this section. When a loop and recursion would both work equally well, reach for the loop; it's usually easier to read and cheaper to run.

One more example worth seeing, because it shows recursion's dark side too:

In [36]:
def fibonacci(n):
    if n <= 1:                              # base case - the first two numbers
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)    # each number is the sum of the two before it

for i in range(8):
    print(fibonacci(i), end=" ")

0 1 1 2 3 5 8 13 

This works, but watch what it's actually doing: `fibonacci(5)` calls `fibonacci(4)` and `fibonacci(3)` — and `fibonacci(4)` *also* calls `fibonacci(3)` separately. The same smaller answers get recalculated over and over, and the number of calls explodes as `n` grows. It's correct, and it's slow. Fixing that (without giving up recursion) is a lesson for later in the course — for now, just recognise that recursion being elegant and recursion being efficient are two different questions.

### Real use case: a P&L with nested expense categories

A **P&L** (profit and loss statement) shows what a business earned, what it spent, and what's left over as profit. Expenses are grouped into categories, which have sub-categories, which can have their own — and every business nests them differently.

You can't know in advance how deep the nesting goes, so you can't write the right number of loops. Recursion doesn't care: every category is handled the same way, however deep. Below, a single expense is just a number, and a category is a dictionary of whatever it contains.

In [ ]:
expenses = {
    "rent": 80000,
    "salaries": {
        "cashiers": 120000,
        "manager": 90000,
    },
    "marketing": {
        "online": {
            "facebook_ads": 25000,
            "google_ads": 15000,
        },
        "flyers": 5000,
    },
    "utilities": 18000,
}

def total_cost(item):
    if type(item) != dict:            # base case - a single expense, just return it
        return item
    amount = 0
    for child in item.values():       # a category - add up everything inside it
        amount += total_cost(child)
    return amount

revenue = 500000
total_expenses = total_cost(expenses)

print("Revenue:       ", revenue)
print("Total expenses:", total_expenses)
print("Profit:        ", revenue - total_expenses)
print()
print("Marketing alone:", total_cost(expenses["marketing"]))

The same function works at **any level** — hand it just `expenses["marketing"]` and you get that category's subtotal, with no extra code.

Plenty of business data has this nested shape: an online store's product categories (Electronics → Phones → Android), a company's org chart, or the JSON an API sends back (Day 18). Whenever data contains smaller copies of itself, recursion is the natural tool.

---
# 12. Decorators — adding behaviour to a function without changing it

A decorator wraps extra behaviour — printing a log, timing, checking a login — around a function, without touching the function's own code. To get there, you need two ideas first.

### Idea 1: a function is a value

A function can be stored in a variable and passed around, like a number or a list. You already did this: `key=lambda ...` handed a function to `.sort()`.

In [ ]:
def shout(text):
    return text.upper()

yell = shout             # no parentheses - not calling it, just giving it a second name
print(yell("hello"))

### Idea 2: a function can create and return another function

In [44]:
def make_greeter(greeting):
    def greet(name):                     # a function defined inside a function
        return f"{greeting}, {name}!"
    return greet                         # hand back the inner function, not its result

say_salaam = make_greeter("Salaam")
say_hello = make_greeter("Hello")

print(say_salaam("Ali"))
print(say_hello("Fatima"))

Salaam, Ali!
Hello, Fatima!


`greet` remembers `greeting` even after `make_greeter` has finished running. A function that remembers values from where it was created is called a **closure** — and it's exactly what makes decorators work.

### Putting it together: a decorator

A decorator is a function that **takes a function, and returns a new version of it** with extra behaviour wrapped around the original.

In [45]:
def announce(func):
    def wrapper():
        print("About to run...")
        func()                       # run the original function
        print("Finished.")
    return wrapper

def make_tea():
    print("Making tea")

make_tea = announce(make_tea)        # replace make_tea with the wrapped version
make_tea()

About to run...
Making tea
Finished.


`make_tea` itself never changed — `announce` built a new function around it. Writing `make_tea = announce(make_tea)` every time is clumsy, so Python gives you a shortcut: put `@announce` on the line above the `def`.

In [ ]:
@announce                            # exactly the same as: make_coffee = announce(make_coffee)
def make_coffee():
    print("Making coffee")

make_coffee()

### Making a decorator work for any function

`announce` only works on functions with no arguments, because `wrapper()` takes none. To wrap *any* function, the wrapper uses `*args` and `**kwargs` — this is where sections 7 and 8 pay off.

In [54]:
def announce(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished.")
        return result                # don't forget to pass the result back
    return wrapper

@announce
def add(a, b):
    return a + b

print(add(3, 4))

Calling add...
add finished.
7


Two new pieces:

- In the `def wrapper(*args, **kwargs)` line, the stars **collect** whatever arguments came in, just like sections 7 and 8.
- In the call `func(*args, **kwargs)`, the stars do the opposite — they **spread** them back out, so the original function receives exactly what the wrapper received.

And `func.__name__` is simply the function's own name, as text.

### Real use case: timing a slow sales report

`import time` loads Python's built-in time tools — imports are covered properly on Day 9. For now, `time.time()` just gives the current time in seconds.

In [ ]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.time() - start:.3f} seconds")
        return result
    return wrapper

@timer
def build_sales_report(order_count):
    revenue = 0
    for order_id in range(order_count):
        revenue += 1500              # pretend every order was worth Rs 1,500
    return revenue

print(build_sales_report(1000000))

### Real use case: only managers can issue refunds

In a POS system, a cashier can ring up sales, but refunds need a manager. Instead of pasting the same permission check into every sensitive function — refunds, big discounts, voiding a sale — write it once as a decorator.

In [ ]:
current_staff = {"name": "Hassan", "role": "cashier"}

def manager_only(func):
    def wrapper(*args, **kwargs):
        if current_staff["role"] != "manager":
            print(f"Denied: {current_staff['name']} is not a manager.")
            return None
        return func(*args, **kwargs)
    return wrapper

@manager_only
def issue_refund(order_id, amount):
    print(f"Refunded Rs {amount} for order {order_id}")

issue_refund("A-1042", 2500)                 # blocked - Hassan is a cashier
current_staff["role"] = "manager"
issue_refund("A-1042", 2500)                 # allowed

### Real use case: an audit log of every sale

A business must be able to answer "who sold what, and for how much?" A decorator can record every call to a function automatically — so nobody can forget to log a sale.

In [ ]:
audit_log = []

def log_transaction(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        audit_log.append(f"{func.__name__}{args} -> {result}")
        return result
    return wrapper

@log_transaction
def sell(item, price, quantity):
    return price * quantity

sell("Mouse", 1500, 2)
sell("Keyboard", 3500, 1)

for entry in audit_log:
    print(entry)

> **Where you'll meet decorators for real:** web frameworks use them to connect a web address to a function (Flask's `@app.route("/home")`), and testing and AI-tooling libraries use them to register functions. You'll *use* ready-made decorators far more often than you'll write your own — but now you know what the `@` is doing.

---
# 13. Generators — producing values one at a time

A normal function builds its whole answer and hands it back all at once with `return`. A **generator** hands values out **one at a time**, only when asked, using `yield` instead of `return`.

In [51]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i       # hand out one value, then pause right here
        i += 1

for number in count_up_to(5):
    print(number)

1
2
3
4
5


`yield` is like a `return` that doesn't end the function — it hands out a value and **pauses**. Next time a value is asked for, the function resumes exactly where it left off, with all its variables intact.

Calling a generator function doesn't run its body at all. It gives you a generator object, and `next()` pulls one value out at a time:

In [47]:
gen = count_up_to(3)
print(gen)            # a generator object - no code inside has run yet
print(next(gen))      # runs until the first yield
print(next(gen))      # resumes, runs to the next yield
print(next(gen))

<generator object count_up_to at 0x1083aa680>
1
2
3


In [48]:
# BREAKS ON PURPOSE - the generator has nothing left to give
print(next(gen))

StopIteration: 

**StopIteration** — the generator ran out of values. A `for` loop quietly watches for this signal and stops, which is why looping over a generator never shows the error.

A generator can also only be walked through **once**:

In [52]:
# WRONG ON PURPOSE - the second list is empty, because the generator is used up
numbers = count_up_to(3)
print(list(numbers))
print(list(numbers))

[1, 2, 3]
[]


### Why bother? Memory.

A list stores every value at once. A generator only ever holds one value at a time. Swap the square brackets of a list comprehension (Day 3's sneak peek) for round brackets and you get a **generator expression**:

In [53]:
import sys

squares_list = [n * n for n in range(1000000)]     # builds all million values now
squares_gen = (n * n for n in range(1000000))      # builds each one only when asked

print("List:     ", sys.getsizeof(squares_list), "bytes")
print("Generator:", sys.getsizeof(squares_gen), "bytes")

List:      8448728 bytes
Generator: 200 bytes


Millions of bytes against a couple of hundred — and the generator stays that size no matter how big the range gets.

### Real use case: sending orders in batches

An online store might need to hand 5,000 orders to a courier's system — but the courier only accepts 100 at a time. AI work looks the same: sending records to a model in chunks rather than all at once. A generator hands out one batch at a time.

In [ ]:
def batches(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]

orders = ["A-1001", "A-1002", "A-1003", "A-1004", "A-1005", "A-1006", "A-1007"]

for batch in batches(orders, 3):
    print("Sending to courier:", batch)

### Real use case: invoice numbers

Every sale needs its own invoice number, forever. Because a generator only runs when asked, it can safely contain a `while True:` loop — it never runs away on its own.

In [ ]:
def invoice_numbers():
    number = 1
    while True:                         # infinite ON PURPOSE - safe, it only runs when asked
        yield f"INV-{number:04d}"       # :04d pads the number to four digits: 0001, 0002...
        number += 1

invoices = invoice_numbers()
print(next(invoices))
print(next(invoices))
print(next(invoices))

> **Where you'll meet generators for real:** Python reads huge files line by line this way, without loading the whole file into memory. And when an AI chatbot streams its reply to you word by word in Month 2, the code receiving that reply is looping over values arriving one at a time — the same idea.

---
# 14. Putting it all together: a day at a small shop

Everything from today, working together in one small program. A computer shop's POS till rings up sales, numbers every receipt, and logs each sale automatically. At closing time, it produces the day's P&L.

Look for each tool as you read: a **generator** for receipt numbers, a **decorator** that logs sales, `*args` for any number of items, a **default argument** for the discount, a **lambda** for the report, and **recursion** for the nested expenses.

In [ ]:
catalogue = {
    "Mouse":    {"price": 1500,  "cost": 900},       # what we sell it for, what we paid for it
    "Keyboard": {"price": 3500,  "cost": 2200},
    "Monitor":  {"price": 45000, "cost": 36000},
}

sales_log = []

def receipt_numbers():                               # generator
    n = 1
    while True:
        yield f"R-{n:03d}"
        n += 1

receipts = receipt_numbers()

def record_sale(func):                               # decorator
    def wrapper(*args, **kwargs):
        sale = func(*args, **kwargs)
        sales_log.append(sale)
        return sale
    return wrapper

@record_sale
def checkout(*items, discount=0):                    # *args, plus a default argument
    revenue = 0
    cost = 0
    for item in items:
        revenue += catalogue[item]["price"]
        cost += catalogue[item]["cost"]
    revenue = revenue * (1 - discount / 100)         # discount is a percentage
    return {"receipt": next(receipts), "items": items, "revenue": revenue, "cost": cost}

One new detail in `checkout(*items, discount=0)`: any parameter written **after** `*items` can only be given by name — `discount=10` — because `*items` swallows every plain value you pass. That's exactly what you want here: no one can accidentally pass a discount as if it were an item.

Now the shop opens, and three customers come in:

In [ ]:
print(checkout("Mouse", "Keyboard"))
print(checkout("Monitor", discount=10))          # a regular customer gets 10% off
print(checkout("Mouse", "Mouse", "Mouse"))

> **Run the setup cell again before re-running this one.** `sales_log` lives outside the functions, so running the checkout cell twice records the same three sales twice — a small, real example of why section 9 warned about shared global state.

Closing time. First, the day's sales from biggest to smallest, then the P&L:

- **Revenue** — money taken from customers
- **Cost of goods sold** — what the shop paid for the items it sold
- **Gross profit** — revenue minus cost of goods sold
- **Operating expenses** — the costs of running the shop at all: rent, staff, electricity
- **Net profit** — what's actually left: gross profit minus operating expenses

In [ ]:
print("Sales, biggest first:")
for sale in sorted(sales_log, key=lambda s: s["revenue"], reverse=True):     # lambda
    print(" ", sale["receipt"], sale["revenue"])

shop_expenses = {                                    # nested - recursion handles it
    "rent": 3000,
    "staff": {"cashier": 2500, "cleaner": 800},
    "utilities": {"electricity": 900, "internet": 300},
}

def total_cost(item):                                # recursion
    if type(item) != dict:
        return item
    amount = 0
    for child in item.values():
        amount += total_cost(child)
    return amount

revenue = 0
cost_of_goods = 0
for sale in sales_log:
    revenue += sale["revenue"]
    cost_of_goods += sale["cost"]

gross_profit = revenue - cost_of_goods
operating_expenses = total_cost(shop_expenses)
net_profit = gross_profit - operating_expenses

print()
print("--- P&L for today ---")
print("Revenue:           ", revenue)
print("Cost of goods sold:", cost_of_goods)
print("Gross profit:      ", gross_profit)
print("Operating expenses:", operating_expenses)
print("Net profit:        ", net_profit)

Rs 50,000 came through the till, but the shop only kept Rs 700 of it. Most of the money went on buying the stock it sold — and almost all of that came from the Monitor sale, which had a thin margin *and* a discount. That's the kind of question a P&L exists to answer, and it took a handful of small functions to produce one.

---
# 15. Practice set: three interview classics

### A calculator function

In [ ]:
def calculate(a, b, operation):
    if operation == "+":
        return a + b
    elif operation == "-":
        return a - b
    elif operation == "*":
        return a * b
    elif operation == "/":
        if b == 0:
            return "Cannot divide by zero"
        return a / b
    else:
        return "Unknown operation"

print(calculate(10, 5, "+"))
print(calculate(10, 5, "/"))
print(calculate(10, 0, "/"))
print(calculate(10, 5, "^"))

### A prime checker

A **prime number** is an integer greater than `1` that has exactly two factors: `1` and itself.

The `is_prime(n)` function checks this step by step:

- If `n < 2`, it returns `False` because `0` and `1` are not prime.
- It tests every number from `2` to `n - 1`.
- If `n % i == 0`, then `n` divides evenly by another number, so it is not prime.
- If no divisor is found, it returns `True`.

For example, `7` is prime because it is not evenly divisible by `2`, `3`, `4`, `5`, or `6`. However, `8` is not prime because `8 % 2 == 0`.

The loop checks numbers from `1` through `19` and prints only those for which `is_prime(number)` returns `True`.

In [ ]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

for number in range(1, 20):
    if is_prime(number):
        print(number, end=" ")

The moment a factor turns up, `return False` leaves the function immediately — no need to keep checking. Same "stop as soon as you know the answer" instinct as `break` on Day 3, just via `return` instead.

> This checks every number up to `n - 1`. A faster version only needs to check up to the square root of `n` — not worth the complexity at these sizes, but worth knowing it exists once your numbers get large.

---
# 16. Your turn

**1.** Write a function `square(n)` that returns `n` squared. Print `square(5)` and `square(12)`.

**2.** Write a function `full_name(first, last)` that returns `"first last"`. Call it once with keyword arguments, in reversed order (`last=` before `first=`).

**3.** Write a function `power(base, exponent=2)` that returns `base` raised to `exponent`, defaulting to squaring. Call it once with just a base, and once with both arguments.

**4.** Write a function `total(*numbers)` that returns the sum of however many numbers you pass it. Test it with zero, one, and five numbers.

**5.** Predict, then run:
```python
def add_item(item, cart=[]):
    cart.append(item)
    return cart

print(add_item("pen"))
print(add_item("book"))
```

**6.** Find the bug:
```python
count = 0

def increment():
    count = count + 1
    return count

print(increment())
```

**7.** Given `[("Ali", 22), ("Zara", 19), ("Hamza", 25)]`, sort it by age using `.sort()` and a `lambda`, then print it.

**8.** Write a recursive function `count_letters(word)` that returns how many letters a word has, without using `len()`. What is your base case?

**9.** Write `sum_digits(n)` recursively so that `sum_digits(1234)` returns `10`. (Hint: `n % 10` gives the last digit, `n // 10` drops it — you already have this one above, but write it yourself before checking.)

**10.** Write `is_prime(n)`, then use it to print every prime number between 1 and 30.

**11.** Write a decorator `shout` that makes any function which returns text return it in UPPERCASE instead. Test it on a function `greet(name)` that returns `f"hello {name}"`.

**12.** Write a generator `evens(limit)` that yields the even numbers from 2 up to `limit`. Loop over `evens(10)` and print each one.

In [ ]:
# Your practice space

### Solutions

Try the exercises yourself first — these are one way to solve them.

In [ ]:
# 1. Square
def square(n):
    return n ** 2

print(square(5))
print(square(12))

In [ ]:
# 2. Full name, keyword arguments in reversed order
def full_name(first, last):
    return f"{first} {last}"

print(full_name(last="Khan", first="Ali"))

In [ ]:
# 3. power() with a default exponent
def power(base, exponent=2):
    return base ** exponent

print(power(5))
print(power(2, 10))

In [ ]:
# 4. total() with any number of arguments
def total(*numbers):
    return sum(numbers)

print(total())
print(total(7))
print(total(1, 2, 3, 4, 5))

In [ ]:
# 5. The mutable default trap - the SAME list is reused every call
def add_item(item, cart=[]):
    cart.append(item)
    return cart

print(add_item("pen"))     # ['pen']
print(add_item("book"))    # ['pen', 'book'] - not just ['book']

In [ ]:
# 6. Fix - reading a global that's also assigned to needs the `global` keyword
count = 0

def increment():
    global count
    count = count + 1
    return count

print(increment())
print(increment())

In [ ]:
# 7. Sort by age with a lambda
people = [("Ali", 22), ("Zara", 19), ("Hamza", 25)]
people.sort(key=lambda person: person[1])
print(people)

In [ ]:
# 8. Recursive letter count - base case is the empty string
def count_letters(word):
    if word == "":                        # base case - no letters left
        return 0
    return 1 + count_letters(word[1:])    # one letter, plus the count of the rest

print(count_letters("python"))
print(count_letters(""))

In [ ]:
# 9. Recursive digit sum - base case is a single digit
def sum_digits(n):
    if n < 10:
        return n
    return n % 10 + sum_digits(n // 10)

print(sum_digits(1234))

In [ ]:
# 10. Prime checker, then use it
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

for number in range(1, 31):
    if is_prime(number):
        print(number, end=" ")

In [ ]:
# 11. A decorator that uppercases the result
def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout
def greet(name):
    return f"hello {name}"

print(greet("ali"))

In [ ]:
# 12. A generator of even numbers
def evens(limit):
    n = 2
    while n <= limit:
        yield n
        n += 2

for number in evens(10):
    print(number)

---
### Today you learned

- `def name(parameters):` defines a function; nothing runs until you **call** it
- `return` sends a value back and exits immediately; no `return` means the function silently gives back `None`
- Arguments are matched **by position**, unless you name them as **keyword arguments** — then order stops mattering
- Default arguments (`name="Hello"`) let callers skip a value — but **never default to a mutable value** like `[]`
- `*args` collects extra positional arguments into a tuple; `**kwargs` collects extra keyword arguments into a dictionary
- A variable created inside a function is **local** and disappears when the function returns; reading a global works, but *assigning* to one needs the `global` keyword
- `lambda parameters: expression` is a one-line, unnamed function — mainly useful as a throwaway `key=` argument, not as a replacement for `def`
- Recursion needs a **base case** and a **recursive case** that moves toward it; it's the natural tool for nested data like folders
- A **decorator** takes a function and returns a wrapped version with extra behaviour; `@name` above a `def` is the shortcut
- A **generator** uses `yield` to hand out values one at a time, only when asked — it saves memory and can even run forever safely

**Next:** putting functions to work with `map()` and `filter()`, and organising code into modules.